In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import warnings
warnings.filterwarnings("ignore")

import time
import json
import logging
from collections import Counter

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam, SGD, RMSprop, Adagrad
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit, ParameterGrid
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from tqdm import tqdm

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.10.1


In [2]:
# =========================
# KONFIGURASI PATH
# =========================
DATA_DIR = "Data"  # folder train/val pool (13 kelas)
EXPORTED_TEST_DIR = "exported_test_set"  # test set locked dari run sebelumnya

# =========================
# KONFIGURASI UMUM
# =========================
IMAGE_SIZE = (224, 224)
BATCH_SIZE_GLOBAL = 16  # lo bilang global batch = 16 (untuk beberapa proses)
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# KONFIGURASI LOOPING LANJUTAN
# =========================
START_ITERATION_INDEX = 6   # lanjut dari sebelumnya (1-5)
N_ITERATIONS = 5            # run 5 iterasi lagi: 6..10
SAMPLING_SIZE = 0.75        # sampling dari trainval_pool per iterasi
VAL_SPLIT_IN_ITER = 0.2     # split dari sample menjadi train_i / val_i

# =========================
# KONFIGURASI TRAINING
# =========================
PHASE1_MAX_EPOCHS = 15
PHASE1_PATIENCE = 3

PHASE2_MAX_EPOCHS = 35
PHASE2_PATIENCE = 5

TUNING_EPOCHS = 5  # lo turunin jadi 5

UNFREEZE_LAST_N = 30  # fine tuning last 30 layer

# =========================
# RUANG HYPERPARAMETER
# =========================
HYPERPARAMETER_SPACE = {
    "learning_rate": [0.001, 0.01],
    "batch_size": [16, 32],
    "dropout_rate": [0.2, 0.3, 0.5],
}

TUNING_CONFIG = {
    "grid_search": 12,
    "random_search": 6,
    "bayesian_optimization": 6,
    "pso": {"n_particles": 4, "n_iterations": 4},
}

OPTIMIZERS = ["Adam", "SGD", "RMSprop", "Adagrad"]
TUNING_METHODS = ["Grid Search", "Random Search", "Bayesian Optimization", "PSO"]

# =========================
# LOGGING
# =========================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("training_log_part2.txt"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# GPU config
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logger.info(f"GPU terdeteksi: {len(gpus)} device(s)")
    except RuntimeError as e:
        logger.warning(f"GPU Error: {e}")
else:
    logger.warning("GPU tidak terdeteksi, menggunakan CPU")

2026-01-14 17:03:26,878 - INFO - GPU terdeteksi: 1 device(s)


In [3]:
# =========================
# DATA LOADING
# =========================
def list_image_files(folder):
    exts = (".png", ".jpg", ".jpeg", ".bmp")
    return [f for f in os.listdir(folder) if f.lower().endswith(exts)]

def load_dataset_from_folder(root_dir):
    """
    Expect struktur:
    root_dir/
      ClassA/
        img1.jpg ...
      ClassB/
        ...
    Return: paths, labels(int), class_names(sorted)
    """
    class_names = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
    paths, labels = [], []

    for idx, cls in enumerate(class_names):
        cls_dir = os.path.join(root_dir, cls)
        imgs = list_image_files(cls_dir)
        for fn in imgs:
            paths.append(os.path.join(cls_dir, fn))
            labels.append(idx)

    return np.array(paths), np.array(labels), class_names

In [4]:
def remove_overlap_by_class_filename(train_paths, train_labels, train_class_names, test_paths, test_class_names):
    """
    Mencegah overlap data train-val dengan test berdasarkan pasangan (class_name, filename).
    Ini cukup robust kalau exported_test_set berisi file yang sama (copied) dengan nama sama.
    """
    # bangun set test: (class, filename)
    test_set = set()
    for p in test_paths:
        cls = os.path.basename(os.path.dirname(p))
        fn = os.path.basename(p)
        test_set.add((cls, fn))

    keep_mask = []
    removed = 0
    for p in train_paths:
        cls = os.path.basename(os.path.dirname(p))
        fn = os.path.basename(p)
        if (cls, fn) in test_set:
            keep_mask.append(False)
            removed += 1
        else:
            keep_mask.append(True)

    keep_mask = np.array(keep_mask, dtype=bool)
    return train_paths[keep_mask], train_labels[keep_mask], removed

In [5]:
# Load TRAINVAL POOL dari Data/
all_paths, all_labels, CLASS_NAMES = load_dataset_from_folder(DATA_DIR)
NUM_CLASSES = len(CLASS_NAMES)

logger.info(f"Loaded train/val pool dari {DATA_DIR}: {len(all_paths)} gambar, {NUM_CLASSES} kelas")
logger.info(f"Class names (train pool): {CLASS_NAMES}")

# Load TEST LOCKED dari exported_test_set/
test_paths, test_labels_raw, TEST_CLASS_NAMES = load_dataset_from_folder(EXPORTED_TEST_DIR)
logger.info(f"Loaded TEST LOCKED dari {EXPORTED_TEST_DIR}: {len(test_paths)} gambar, {len(TEST_CLASS_NAMES)} kelas")
logger.info(f"Class names (test): {TEST_CLASS_NAMES}")

2026-01-14 17:03:26,940 - INFO - Loaded train/val pool dari Data: 2462 gambar, 13 kelas
2026-01-14 17:03:26,940 - INFO - Class names (train pool): ['Betawi', 'Bokorkencor', 'Buketan', 'Dayak', 'Jlamprang', 'Kawung', 'Liong', 'Megamendung', 'Parang', 'Sekarjagad', 'Sidoluhur', 'Sidomukti', 'Singobarong']
2026-01-14 17:03:26,950 - INFO - Loaded TEST LOCKED dari exported_test_set: 370 gambar, 13 kelas
2026-01-14 17:03:26,950 - INFO - Class names (test): ['Betawi', 'Bokorkencor', 'Buketan', 'Dayak', 'Jlamprang', 'Kawung', 'Liong', 'Megamendung', 'Parang', 'Sekarjagad', 'Sidoluhur', 'Sidomukti', 'Singobarong']


In [6]:
# Validasi class order harus sama, biar label mapping konsisten
if CLASS_NAMES != TEST_CLASS_NAMES:
    raise ValueError(
        "Urutan/nama kelas di Data/ dan exported_test_set/ tidak sama.\n"
        f"Data/: {CLASS_NAMES}\n"
        f"Test/: {TEST_CLASS_NAMES}\n"
        "Samakan folder class-nya (nama & urutan sorting) supaya label konsisten."
    )

In [7]:
# Buang overlap
all_paths_no_overlap, all_labels_no_overlap, removed_n = remove_overlap_by_class_filename(
    all_paths, all_labels, CLASS_NAMES, test_paths, TEST_CLASS_NAMES
)
logger.info(f"Overlap yang dibuang dari train/val pool: {removed_n} file")

X_trainval_pool = all_paths_no_overlap
y_trainval_pool = all_labels_no_overlap

X_test = test_paths
y_test = test_labels_raw

logger.info(f"Final TrainVal Pool: {len(X_trainval_pool)} | Final Test Locked: {len(X_test)}")

logger.info("Distribusi Test Locked:")
dist_test = Counter(y_test)
for k in range(NUM_CLASSES):
    logger.info(f"  {CLASS_NAMES[k]} = {dist_test.get(k, 0)}")

2026-01-14 17:03:27,017 - INFO - Overlap yang dibuang dari train/val pool: 370 file
2026-01-14 17:03:27,017 - INFO - Final TrainVal Pool: 2092 | Final Test Locked: 370
2026-01-14 17:03:27,017 - INFO - Distribusi Test Locked:
2026-01-14 17:03:27,017 - INFO -   Betawi = 9
2026-01-14 17:03:27,017 - INFO -   Bokorkencor = 9
2026-01-14 17:03:27,017 - INFO -   Buketan = 9
2026-01-14 17:03:27,017 - INFO -   Dayak = 9
2026-01-14 17:03:27,017 - INFO -   Jlamprang = 9
2026-01-14 17:03:27,017 - INFO -   Kawung = 96
2026-01-14 17:03:27,017 - INFO -   Liong = 9
2026-01-14 17:03:27,028 - INFO -   Megamendung = 90
2026-01-14 17:03:27,028 - INFO -   Parang = 95
2026-01-14 17:03:27,028 - INFO -   Sekarjagad = 8
2026-01-14 17:03:27,028 - INFO -   Sidoluhur = 9
2026-01-14 17:03:27,028 - INFO -   Sidomukti = 9
2026-01-14 17:03:27,028 - INFO -   Singobarong = 9


In [8]:
# Save test info (opsional, buat audit)
with open("test_set_info_part2.json", "w") as f:
    json.dump(
        {
            "test_dir": EXPORTED_TEST_DIR,
            "test_count": int(len(X_test)),
            "class_names": CLASS_NAMES,
            "distribution": {CLASS_NAMES[k]: int(dist_test.get(k, 0)) for k in range(NUM_CLASSES)},
        },
        f,
        indent=2
    )

In [9]:
# =========================
# DATA AUGMENTATION
# brightness udah dihapus sesuai instruksi lo
# =========================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(rescale=1./255)

In [10]:
# =========================
# UTIL: MODEL + DATA GENERATOR
# =========================
def build_model(dropout_rate=0.3):
    base_model = MobileNetV2(
        input_shape=IMAGE_SIZE + (3,),
        include_top=False,
        weights="imagenet"
    )
    base_model.trainable = False

    inputs = Input(shape=IMAGE_SIZE + (3,), name="input_layer")
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D(name="global_avg_pooling")(x)
    x = BatchNormalization(name="batch_norm")(x)
    x = Dropout(dropout_rate, name="dropout")(x)
    outputs = Dense(NUM_CLASSES, activation="softmax", name="output_layer")(x)

    model = Model(inputs, outputs, name="MobileNetV2_Batik_13Class")
    return model, base_model

In [11]:
def load_batch_data(X, y, batch_size, datagen):
    indices = np.arange(len(X))
    while True:
        np.random.shuffle(indices)
        for start in range(0, len(X), batch_size):
            end = min(start + batch_size, len(X))
            batch_idx = indices[start:end]

            batch_images = []
            batch_labels = []

            for idx in batch_idx:
                try:
                    img = tf.keras.utils.load_img(X[idx], target_size=IMAGE_SIZE)
                    if img.mode != "RGB":
                        img = img.convert("RGB")

                    arr = tf.keras.utils.img_to_array(img)
                    arr = datagen.standardize(arr)
                    batch_images.append(arr)
                    batch_labels.append(y[idx])
                except:
                    continue

            if len(batch_images) > 0:
                yield np.array(batch_images), tf.keras.utils.to_categorical(batch_labels, NUM_CLASSES)

In [12]:
def get_optimizer(optimizer_name, learning_rate):
    if optimizer_name == "Adam":
        return Adam(learning_rate=learning_rate)
    if optimizer_name == "SGD":
        return SGD(learning_rate=learning_rate, momentum=0.9, nesterov=True)
    if optimizer_name == "RMSprop":
        return RMSprop(learning_rate=learning_rate)
    if optimizer_name == "Adagrad":
        return Adagrad(learning_rate=learning_rate)
    raise ValueError(f"Unknown optimizer: {optimizer_name}")

In [13]:
def freeze_batchnorm_layers(model):
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

In [14]:
# =========================
# TWO PHASE TRAINING (training time tidak jadi metrik utama)
# =========================
def two_phase_training(optimizer_name, best_params, X_train, y_train, X_val, y_val, iteration, experiment_name):
    model, base_model = build_model(dropout_rate=float(best_params["dropout_rate"]))

    bs = int(best_params["batch_size"])
    steps = max(1, len(X_train) // bs)
    val_steps = max(1, len(X_val) // bs)

    # Phase 1
    base_model.trainable = False
    opt1 = get_optimizer(optimizer_name, float(best_params["learning_rate"]))
    model.compile(optimizer=opt1, loss="categorical_crossentropy", metrics=["accuracy"])

    es1 = EarlyStopping(
        monitor="val_loss",
        patience=PHASE1_PATIENCE,
        min_delta=0.001,
        restore_best_weights=True,
        verbose=0
    )

    h1 = model.fit(
        load_batch_data(X_train, y_train, bs, train_datagen),
        steps_per_epoch=steps,
        validation_data=load_batch_data(X_val, y_val, bs, val_datagen),
        validation_steps=val_steps,
        epochs=PHASE1_MAX_EPOCHS,
        callbacks=[es1],
        verbose=0
    )

    # Phase 2
    base_model.trainable = True
    for layer in base_model.layers[:-UNFREEZE_LAST_N]:
        layer.trainable = False

    freeze_batchnorm_layers(model)

    fine_lr = float(best_params["learning_rate"]) / 10.0
    opt2 = get_optimizer(optimizer_name, fine_lr)
    model.compile(optimizer=opt2, loss="categorical_crossentropy", metrics=["accuracy"])

    es2 = EarlyStopping(
        monitor="val_loss",
        patience=PHASE2_PATIENCE,
        min_delta=0.001,
        restore_best_weights=True,
        verbose=0
    )

    h2 = model.fit(
        load_batch_data(X_train, y_train, bs, train_datagen),
        steps_per_epoch=steps,
        validation_data=load_batch_data(X_val, y_val, bs, val_datagen),
        validation_steps=val_steps,
        epochs=PHASE2_MAX_EPOCHS,
        callbacks=[es2],
        verbose=0
    )

    history = {
        "accuracy": h1.history.get("accuracy", []) + h2.history.get("accuracy", []),
        "val_accuracy": h1.history.get("val_accuracy", []) + h2.history.get("val_accuracy", []),
        "loss": h1.history.get("loss", []) + h2.history.get("loss", []) ,
        "val_loss": h1.history.get("val_loss", []) + h2.history.get("val_loss", []),
    }

    best_val_acc = float(np.max(history["val_accuracy"])) if history["val_accuracy"] else np.nan
    best_val_loss = float(np.min(history["val_loss"])) if history["val_loss"] else np.nan
    epochs_run = int(len(history["val_loss"]))

    return model, history, best_val_acc, best_val_loss, epochs_run

In [15]:
# =========================
# STRATIFIED SAMPLING
# =========================
def stratified_sample(X, y, sampling_size, seed):
    sss = StratifiedShuffleSplit(n_splits=1, train_size=sampling_size, random_state=seed)
    idx_sample, idx_rest = next(sss.split(X, y))
    return X[idx_sample], y[idx_sample], X[idx_rest], y[idx_rest]

In [16]:
# =========================
# TEST EVALUATION (habiskan semua data)
# =========================
def evaluate_on_test_full(model, X_test, y_test, batch_size):
    gen = load_batch_data(X_test, y_test, batch_size, val_datagen)

    # jumlah batch = ceil(n / bs) supaya tidak ada sisa yang hilang
    n = len(X_test)
    steps = int(np.ceil(n / batch_size))

    y_true = []
    y_pred = []

    for _ in tqdm(range(steps), desc="Testing"):
        x_batch, y_batch = next(gen)
        probs = model.predict(x_batch, verbose=0)
        y_true.extend(np.argmax(y_batch, axis=1))
        y_pred.extend(np.argmax(probs, axis=1))

    # mungkin generator terakhir ngasih batch yang “lebih kecil”, tapi aman
    return np.array(y_true)[:n], np.array(y_pred)[:n]

In [17]:
# =========================
# TUNING METHODS (return: best_params, best_score, search_time_sec)
# search_time = waktu hyperparameter searching
# =========================
def grid_search_tuning(optimizer_name, X_train, y_train, X_val, y_val, iteration):
    start = time.time()

    param_grid = list(ParameterGrid(HYPERPARAMETER_SPACE))
    best_score = -1.0
    best_params = None

    for params in tqdm(param_grid, desc=f"Grid Search {optimizer_name} (Iter {iteration})"):
        model, base_model = build_model(dropout_rate=float(params["dropout_rate"]))
        opt = get_optimizer(optimizer_name, float(params["learning_rate"]))
        model.compile(optimizer=opt, loss="categorical_crossentropy", metrics=["accuracy"])

        es = EarlyStopping(monitor="val_loss", patience=2, min_delta=0.001, restore_best_weights=True, verbose=0)

        bs = int(params["batch_size"])
        steps = max(1, len(X_train) // bs)
        val_steps = max(1, len(X_val) // bs)

        hist = model.fit(
            load_batch_data(X_train, y_train, bs, train_datagen),
            steps_per_epoch=steps,
            validation_data=load_batch_data(X_val, y_val, bs, val_datagen),
            validation_steps=val_steps,
            epochs=TUNING_EPOCHS,
            callbacks=[es],
            verbose=0
        )

        val_acc = float(np.max(hist.history.get("val_accuracy", [0.0])))

        if val_acc > best_score:
            best_score = val_acc
            best_params = params

        del model, base_model, hist
        tf.keras.backend.clear_session()

    search_time = time.time() - start
    return best_params, best_score, search_time

In [18]:
def random_search_tuning(optimizer_name, X_train, y_train, X_val, y_val, iteration, n_trials=6):
    start = time.time()

    best_score = -1.0
    best_params = None

    for _ in tqdm(range(n_trials), desc=f"Random Search {optimizer_name} (Iter {iteration})"):
        params = {
            "learning_rate": float(np.random.choice(HYPERPARAMETER_SPACE["learning_rate"])),
            "batch_size": int(np.random.choice(HYPERPARAMETER_SPACE["batch_size"])),
            "dropout_rate": float(np.random.choice(HYPERPARAMETER_SPACE["dropout_rate"])),
        }

        model, base_model = build_model(dropout_rate=params["dropout_rate"])
        opt = get_optimizer(optimizer_name, params["learning_rate"])
        model.compile(optimizer=opt, loss="categorical_crossentropy", metrics=["accuracy"])

        es = EarlyStopping(monitor="val_loss", patience=2, min_delta=0.001, restore_best_weights=True, verbose=0)

        bs = int(params["batch_size"])
        steps = max(1, len(X_train) // bs)
        val_steps = max(1, len(X_val) // bs)

        hist = model.fit(
            load_batch_data(X_train, y_train, bs, train_datagen),
            steps_per_epoch=steps,
            validation_data=load_batch_data(X_val, y_val, bs, val_datagen),
            validation_steps=val_steps,
            epochs=TUNING_EPOCHS,
            callbacks=[es],
            verbose=0
        )

        val_acc = float(np.max(hist.history.get("val_accuracy", [0.0])))

        if val_acc > best_score:
            best_score = val_acc
            best_params = params

        del model, base_model, hist
        tf.keras.backend.clear_session()

    search_time = time.time() - start
    return best_params, best_score, search_time

In [19]:
def bayesian_optimization_tuning(optimizer_name, X_train, y_train, X_val, y_val, iteration, max_trials=6):
    import keras_tuner as kt
    import logging as kt_logging
    kt_logging.getLogger("tensorflow").setLevel(kt_logging.ERROR)
    kt_logging.getLogger("keras_tuner").setLevel(kt_logging.WARNING)

    start = time.time()

    def build_tuning_model(hp):
        lr = hp.Choice("learning_rate", HYPERPARAMETER_SPACE["learning_rate"])
        bs = hp.Choice("batch_size", HYPERPARAMETER_SPACE["batch_size"])
        dr = hp.Choice("dropout_rate", HYPERPARAMETER_SPACE["dropout_rate"])

        model, _ = build_model(dropout_rate=float(dr))
        opt = get_optimizer(optimizer_name, float(lr))
        model.compile(optimizer=opt, loss="categorical_crossentropy", metrics=["accuracy"])
        return model

    tuner = kt.BayesianOptimization(
        build_tuning_model,
        objective="val_accuracy",
        max_trials=max_trials,
        directory="tuning_logs_part2",
        project_name=f"bayesian_{optimizer_name}_iter{iteration}",
        overwrite=True
    )

    es = EarlyStopping(
        monitor="val_loss",
        patience=2,
        min_delta=0.001,
        restore_best_weights=True,
        verbose=0
    )

    steps = max(1, len(X_train) // BATCH_SIZE_GLOBAL)
    val_steps = max(1, len(X_val) // BATCH_SIZE_GLOBAL)

    tuner.search(
        load_batch_data(X_train, y_train, BATCH_SIZE_GLOBAL, train_datagen),
        steps_per_epoch=steps,
        validation_data=load_batch_data(X_val, y_val, BATCH_SIZE_GLOBAL, val_datagen),
        validation_steps=val_steps,
        epochs=TUNING_EPOCHS,
        callbacks=[es],
        verbose=0
    )

    # best hyperparameters
    best_hps = tuner.get_best_hyperparameters(1)[0]
    best_params = {
        "learning_rate": float(best_hps.get("learning_rate")),
        "batch_size": int(best_hps.get("batch_size")),
        "dropout_rate": float(best_hps.get("dropout_rate")),
    }

    # score tuning yang kompatibel lintas versi:
    # ambil trial terbaik dari oracle lalu pakai .score
    try:
        best_trial = tuner.oracle.get_best_trials(num_trials=1)[0]
        best_score = float(best_trial.score) if best_trial.score is not None else np.nan
    except Exception:
        best_score = np.nan

    search_time = time.time() - start

    # bersih-bersih
    tf.keras.backend.clear_session()

    return best_params, best_score, search_time


In [20]:
def pso_tuning(optimizer_name, X_train, y_train, X_val, y_val, iteration, n_particles=4, n_iterations=4):
    import pyswarms as ps

    start = time.time()

    lr_options = HYPERPARAMETER_SPACE["learning_rate"]
    bs_options = HYPERPARAMETER_SPACE["batch_size"]
    dr_options = HYPERPARAMETER_SPACE["dropout_rate"]

    bounds = (
        np.array([0, 0, 0]),
        np.array([len(lr_options)-1, len(bs_options)-1, len(dr_options)-1])
    )

    def objective_function(particles):
        costs = []
        for pos in particles:
            lr_idx = int(np.clip(np.round(pos[0]), 0, len(lr_options)-1))
            bs_idx = int(np.clip(np.round(pos[1]), 0, len(bs_options)-1))
            dr_idx = int(np.clip(np.round(pos[2]), 0, len(dr_options)-1))

            params = {
                "learning_rate": float(lr_options[lr_idx]),
                "batch_size": int(bs_options[bs_idx]),
                "dropout_rate": float(dr_options[dr_idx]),
            }

            model, base_model = build_model(dropout_rate=params["dropout_rate"])
            opt = get_optimizer(optimizer_name, params["learning_rate"])
            model.compile(optimizer=opt, loss="categorical_crossentropy", metrics=["accuracy"])

            es = EarlyStopping(monitor="val_loss", patience=2, min_delta=0.001, restore_best_weights=True, verbose=0)

            bs = params["batch_size"]
            steps = max(1, len(X_train) // bs)
            val_steps = max(1, len(X_val) // bs)

            hist = model.fit(
                load_batch_data(X_train, y_train, bs, train_datagen),
                steps_per_epoch=steps,
                validation_data=load_batch_data(X_val, y_val, bs, val_datagen),
                validation_steps=val_steps,
                epochs=TUNING_EPOCHS,
                callbacks=[es],
                verbose=0
            )

            val_acc = float(np.max(hist.history.get("val_accuracy", [0.0])))
            costs.append(1.0 - val_acc)

            del model, base_model, hist
            tf.keras.backend.clear_session()

        return np.array(costs)

    options = {"c1": 0.5, "c2": 0.3, "w": 0.9}
    optimizer_pso = ps.single.GlobalBestPSO(
        n_particles=n_particles,
        dimensions=3,
        options=options,
        bounds=bounds
    )

    cost, best_pos = optimizer_pso.optimize(objective_function, iters=n_iterations, verbose=False)

    lr_idx = int(np.clip(np.round(best_pos[0]), 0, len(lr_options)-1))
    bs_idx = int(np.clip(np.round(best_pos[1]), 0, len(bs_options)-1))
    dr_idx = int(np.clip(np.round(best_pos[2]), 0, len(dr_options)-1))

    best_params = {
        "learning_rate": float(lr_options[lr_idx]),
        "batch_size": int(bs_options[bs_idx]),
        "dropout_rate": float(dr_options[dr_idx]),
    }

    best_score = float(1.0 - cost)

    search_time = time.time() - start
    return best_params, best_score, search_time

In [21]:
# =========================
# MAIN LOOP (5 ITERASI LANJUTAN)
# =========================
all_loop_results = []
best_models_summary = []

os.makedirs("models_per_iter_part2", exist_ok=True)

for it in range(START_ITERATION_INDEX, START_ITERATION_INDEX + N_ITERATIONS):
    logger.info(f"\n{'='*90}\nITERATION {it}\n{'='*90}")

    # 1) sampling dari trainval_pool (tanpa test)
    X_sample, y_sample, _, _ = stratified_sample(
        X_trainval_pool, y_trainval_pool,
        sampling_size=SAMPLING_SIZE,
        seed=SEED + it
    )

    # 2) split sample jadi train_i / val_i
    X_train_i, X_val_i, y_train_i, y_val_i = train_test_split(
        X_sample, y_sample,
        test_size=VAL_SPLIT_IN_ITER,
        stratify=y_sample,
        random_state=SEED + 1000 + it
    )

    logger.info(f"[Iter {it}] Sample={len(X_sample)} | Train_i={len(X_train_i)} | Val_i={len(X_val_i)}")

    dist_train = Counter(y_train_i)
    dist_val = Counter(y_val_i)
    logger.info("[Train_i dist] " + ", ".join([f"{CLASS_NAMES[k]}={dist_train.get(k,0)}" for k in range(NUM_CLASSES)]))
    logger.info("[Val_i dist] " + ", ".join([f"{CLASS_NAMES[k]}={dist_val.get(k,0)}" for k in range(NUM_CLASSES)]))

    iteration_results = []
    best_iter_entry = None

    for optimizer_name in OPTIMIZERS:
        for tuning_method in TUNING_METHODS:
            experiment_name = f"Iter{it}_{optimizer_name}_{tuning_method.replace(' ', '_')}"
            logger.info(f"\n[Iter {it}] EXPERIMENT: {experiment_name}")

            try:
                # 3) tuning step (catat search time)
                if tuning_method == "Grid Search":
                    best_params, tuning_score, search_time = grid_search_tuning(
                        optimizer_name, X_train_i, y_train_i, X_val_i, y_val_i, it
                    )
                elif tuning_method == "Random Search":
                    best_params, tuning_score, search_time = random_search_tuning(
                        optimizer_name, X_train_i, y_train_i, X_val_i, y_val_i, it,
                        n_trials=TUNING_CONFIG["random_search"]
                    )
                elif tuning_method == "Bayesian Optimization":
                    best_params, tuning_score, search_time = bayesian_optimization_tuning(
                        optimizer_name, X_train_i, y_train_i, X_val_i, y_val_i, it,
                        max_trials=TUNING_CONFIG["bayesian_optimization"]
                    )
                elif tuning_method == "PSO":
                    best_params, tuning_score, search_time = pso_tuning(
                        optimizer_name, X_train_i, y_train_i, X_val_i, y_val_i, it,
                        n_particles=TUNING_CONFIG["pso"]["n_particles"],
                        n_iterations=TUNING_CONFIG["pso"]["n_iterations"]
                    )
                else:
                    raise ValueError("Unknown tuning method")

                # 4) train best config (two-phase) - training time TIDAK dicatat sebagai waktu utama
                model, history, best_val_acc_hist, best_val_loss_hist, epochs_run = two_phase_training(
                    optimizer_name, best_params,
                    X_train_i, y_train_i, X_val_i, y_val_i,
                    it, experiment_name
                )

                # 5) evaluate on validation (single pass)
                bs = int(best_params["batch_size"])
                val_steps = max(1, int(np.ceil(len(X_val_i) / bs)))
                val_loss, val_acc = model.evaluate(
                    load_batch_data(X_val_i, y_val_i, bs, val_datagen),
                    steps=val_steps,
                    verbose=0
                )

                result_entry = {
                    "Iteration": it,
                    "Experiment": experiment_name,
                    "Optimizer": optimizer_name,
                    "Tuning Method": tuning_method,
                    "Learning Rate": float(best_params["learning_rate"]),
                    "Batch Size": int(best_params["batch_size"]),
                    "Dropout Rate": float(best_params["dropout_rate"]),
                    "Tuning Score (val_acc)": float(tuning_score) if tuning_score is not None else np.nan,
                    "Val Accuracy (eval)": float(val_acc),
                    "Val Loss (eval)": float(val_loss),
                    "Best Val Accuracy (history)": float(best_val_acc_hist),
                    "Best Val Loss (history)": float(best_val_loss_hist),
                    "Epochs Run": int(epochs_run),
                    "Search Time (s)": float(search_time),
                    "Search Time (min)": float(search_time / 60.0),
                }

                iteration_results.append(result_entry)
                all_loop_results.append(result_entry)

                # pilih best per iterasi berdasar val_acc (eval)
                if (best_iter_entry is None) or (result_entry["Val Accuracy (eval)"] > best_iter_entry["Val Accuracy (eval)"]):
                    best_iter_entry = result_entry.copy()
                    best_path = f"models_per_iter_part2/best_iter{it}.keras"
                    model.save(best_path)
                    best_iter_entry["Saved Model Path"] = best_path

                del model, history
                tf.keras.backend.clear_session()

                logger.info(f"[Iter {it}] OK: val_acc={val_acc:.4f}, val_loss={val_loss:.4f}, search_time_min={search_time/60:.2f}")

            except Exception as e:
                logger.exception(f"[Iter {it}] ERROR in {experiment_name}: {str(e)}")
                tf.keras.backend.clear_session()
                continue

    # save per-iteration results
    iter_df = pd.DataFrame(iteration_results)
    iter_df.to_csv(f"results_iteration_{it}.csv", index=False)
    logger.info(f"[Iter {it}] Results saved: results_iteration_{it}.csv")

    if best_iter_entry is not None:
        best_models_summary.append(best_iter_entry)

2026-01-14 17:03:27,306 - INFO - 
ITERATION 6
2026-01-14 17:03:27,317 - INFO - [Iter 6] Sample=1569 | Train_i=1255 | Val_i=314
2026-01-14 17:03:27,320 - INFO - [Train_i dist] Betawi=30, Bokorkencor=30, Buketan=31, Dayak=29, Jlamprang=30, Kawung=326, Liong=31, Megamendung=307, Parang=323, Sekarjagad=30, Sidoluhur=29, Sidomukti=29, Singobarong=30
2026-01-14 17:03:27,321 - INFO - [Val_i dist] Betawi=8, Bokorkencor=8, Buketan=7, Dayak=7, Jlamprang=8, Kawung=81, Liong=8, Megamendung=77, Parang=81, Sekarjagad=7, Sidoluhur=7, Sidomukti=7, Singobarong=8
2026-01-14 17:03:27,321 - INFO - 
[Iter 6] EXPERIMENT: Iter6_Adam_Grid_Search
Grid Search Adam (Iter 6): 100%|██████████| 12/12 [06:53<00:00, 34.47s/it]
2026-01-14 17:12:13,401 - INFO - [Iter 6] OK: val_acc=0.8885, val_loss=0.3239, search_time_min=6.89
2026-01-14 17:12:13,404 - INFO - 
[Iter 6] EXPERIMENT: Iter6_Adam_Random_Search
Random Search Adam (Iter 6): 100%|██████████| 6/6 [03:04<00:00, 30.79s/it]
2026-01-14 17:17:38,188 - INFO - [Iter 6

In [22]:
# save combined results
all_df = pd.DataFrame(all_loop_results)
all_df.to_csv("all_loop_results_part2.csv", index=False)
logger.info("Saved combined results: all_loop_results_part2.csv")

best_df = pd.DataFrame(best_models_summary)
best_df.to_csv("best_models_summary_part2.csv", index=False)
logger.info("Saved best per iteration: best_models_summary_part2.csv")

print("\nDONE. Files generated:")
print("- results_iteration_{ITER}.csv (ITER = 6..10)")
print("- all_loop_results_part2.csv")
print("- best_models_summary_part2.csv")
print("- models_per_iter_part2/best_iter{ITER}.keras")

2026-01-15 03:48:55,045 - __main__ - INFO - Saved combined results: all_loop_results_part2.csv
2026-01-15 03:48:55,045 - __main__ - INFO - Saved best per iteration: best_models_summary_part2.csv



DONE. Files generated:
- results_iteration_{ITER}.csv (ITER = 6..10)
- all_loop_results_part2.csv
- best_models_summary_part2.csv
- models_per_iter_part2/best_iter{ITER}.keras
